In [1]:
from pathlib import Path
import argparse
import numpy as np
import pandas as pd


In [2]:
path =("Toronto_Island_Ferry_Tickets.csv")


In [3]:
TARGETS = ["Sales Count", "Redemption Count"]

# minutes; data is resampled to 15-minute intervals
HORIZONS = [15, 30, 60, 120]



In [4]:


"""
    Read, clean, aggregate, and resample the original ticket CSV.

    Missing intervals are NOT automatically treated as zero sales.
    Short internal gaps (at most two 15-minute intervals) are interpolated
    for input features. A separate observation flag preserves which
    timestamps existed in the original data.
"""
def load_data(path):
    """Load raw or engineered CSV, aggregate timestamps, and build a 15-min grid."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"CSV not found: {path.resolve()}")
    
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    time_col = next((c for c in ["Timestamp", "Datetime", "DateTime", "datetime", "Date"] if c in df.columns), None)
    
    if time_col is None and "Unnamed: 0" in df.columns:
        time_col = "Unnamed: 0"
    
    if time_col is None:
        raise ValueError(f"Timestamp column not found. Columns: {df.columns.tolist()}")
    
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.dropna(subset=[time_col]).rename(columns={time_col: "Timestamp"})
    rename = {}
    
    for c in df.columns:
        n = c.lower().replace("_", " ").strip()
        if n in {"sales count", "sales", "ticket sales"}: rename[c] = "Sales Count"
        if n in {"redemption count", "redemptions", "redemption"}: rename[c] = "Redemption Count"
    
    df = df.rename(columns=rename)
    
    for c in TARGETS:
        if c not in df.columns:
            raise ValueError(f"Required column '{c}' not found. Columns: {df.columns.tolist()}")
        df[c] = pd.to_numeric(df[c], errors="coerce")
    
    df = df.groupby("Timestamp")[TARGETS].sum(min_count=1).sort_index()
    df = df.resample("15min").sum(min_count=1)
    
    # Mark original observations before interpolating; do not treat missing as zero.
    df["Observed_Interval"] = df[TARGETS].notna().any(axis=1).astype(int)
    df["Missing_Interval"] = 1 - df["Observed_Interval"]
    
    # Fill only short internal gaps (maximum two 15-minute intervals).
    df[TARGETS] = df[TARGETS].interpolate(method="time", limit=2, limit_area="inside")
    return df



In [5]:

"""
    Create past-only features and future targets for every horizon.

    Important:
    - Lag and rolling features only use values available at the forecast origin.
    - Targets are created with shift(-steps).
    - Rows whose future target interval was not observed in the source data
      are marked invalid for scoring/training.
"""
def build_features(df):
    """Create past-only lags, rolling statistics, and calendar features."""
    
    out = df.copy()
    
    for target in TARGETS:
        prefix = target.replace(" ", "_")
        
        for lag in [1, 2, 4, 8, 96]:
            out[f"{prefix}_Lag_{lag}"] = out[target].shift(lag)
        past = out[target].shift(1)  # shift first to prevent current/future leakage
        
        for window in [4, 8, 96]:
            r = past.rolling(window, min_periods=window)
            out[f"{prefix}_RollingMean_{window}"] = r.mean()
            out[f"{prefix}_RollingStd_{window}"] = r.std()
            out[f"{prefix}_RollingMax_{window}"] = r.max()
    
    idx = out.index
    out["Hour"], out["Minute"] = idx.hour, idx.minute
    out["DayOfWeek"], out["Day"], out["Month"] = idx.dayofweek, idx.day, idx.month
    out["Year"], out["Quarter"] = idx.year, idx.quarter
    out["IsWeekend"] = (idx.dayofweek >= 5).astype(int)
    out["Hour_Sin"] = np.sin(2 * np.pi * out["Hour"] / 24)
    out["Hour_Cos"] = np.cos(2 * np.pi * out["Hour"] / 24)
    out["DayOfWeek_Sin"] = np.sin(2 * np.pi * out["DayOfWeek"] / 7)
    out["DayOfWeek_Cos"] = np.cos(2 * np.pi * out["DayOfWeek"] / 7)
    
    return out.replace([np.inf, -np.inf], np.nan)



In [6]:
def get_feature_columns(df):
    """Keep engineered predictors; exclude current counts and future targets."""
    
    return [c for c in df.columns if (
        "_Lag_" in c or "_RollingMean_" in c or "_RollingStd_" in c or "_RollingMax_" in c
        or c in {"Hour", "Minute", "DayOfWeek", "Day", "Month", "Year", "Quarter", "IsWeekend",
                 "Hour_Sin", "Hour_Cos", "DayOfWeek_Sin", "DayOfWeek_Cos",
                 "Observed_Interval", "Missing_Interval"}
    )]





In [7]:

def make_xy(features, target, horizon_minutes):
    """Create predictors and target at +15/+30/+60/+120 minutes."""
    
    if horizon_minutes not in HORIZONS:
        raise ValueError(f"Unsupported horizon {horizon_minutes}; use {HORIZONS}")
    
    steps = horizon_minutes // 15
    X = features[get_feature_columns(features)].copy()
    y = features[target].shift(-steps).rename("target")
    
    # Only evaluate targets from timestamps that existed in the original data.
    target_observed = features["Observed_Interval"].shift(-steps).eq(1)
    valid = X.notna().all(axis=1) & y.notna() & target_observed
    
    return X.loc[valid], y.loc[valid]





In [8]:
def time_split(X, y, test_fraction=0.20):
    """Chronological split: oldest rows train; newest rows test; no shuffle."""
    
    if not 0 < test_fraction < 1: raise ValueError("test_fraction must be between 0 and 1")
    
    order = np.argsort(X.index.values)
    X, y = X.iloc[order], y.iloc[order]
    
    cut = int(len(X) * (1 - test_fraction))
    
    if cut < 1 or cut >= len(X): raise ValueError("Not enough valid data for split")
    return X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]



In [9]:
def rolling_origin_splits(X, initial_fraction=0.60, validation_fraction=0.10, max_splits=4):
    """Expanding-window folds. Every validation block is later than its training block."""
    
    n = len(X); first_train = int(n * initial_fraction); val_size = max(1, int(n * validation_fraction))
    
    if first_train < 1 or first_train + val_size > n: raise ValueError("Not enough rows for rolling validation")
    folds, train_end = [], first_train
    
    while train_end + val_size <= n and len(folds) < max_splits:
        folds.append((np.arange(0, train_end), np.arange(train_end, train_end + val_size)))
        train_end += val_size
    
    if not folds: raise ValueError("Could not form rolling-origin folds")
    
    return folds


In [10]:
def main():
    p = argparse.ArgumentParser()
    
    p.add_argument("--data", default=path)
    
    p.add_argument("--test-fraction", type=float, default=0.20)
    
    p.add_argument("--output-dir", default="forecast_outputs")
    
    args, extra_args = p.parse_known_args()
    if extra_args:
        is_kernel_args = (
            len(extra_args) == 1 and extra_args[0].startswith("--f=")
        ) or (
            len(extra_args) == 2 and extra_args[0] in {"-f", "--f"}
        )
        if not is_kernel_args:
            p.error(f"unrecognized arguments: {' '.join(extra_args)}")
    outdir = Path(args.output_dir); outdir.mkdir(parents=True, exist_ok=True)
    data = build_features(load_data(args.data))
    
    print(f"Timeline: {data.index.min()} to {data.index.max()}")
    print(f"Rows: {len(data):,}; missing intervals: {int(data['Missing_Interval'].sum()):,}")
    
    summaries = []
    
    for target in TARGETS:
        
        for horizon in HORIZONS:
            X, y = make_xy(data, target, horizon)
            Xtr, Xte, ytr, yte = time_split(X, y, args.test_fraction)
            stem = f"{target.replace(' ', '_').lower()}_{horizon}min"
            tr, te = Xtr.copy(), Xte.copy()
            tr["target"] = ytr; te["target"] = yte
            tr.to_csv(outdir / f"{stem}_train.csv", index_label="Timestamp")
            te.to_csv(outdir / f"{stem}_test.csv", index_label="Timestamp")
            
            summaries.append({"target": target, "horizon_minutes": horizon, "valid_rows": len(X),
                              "train_rows": len(Xtr), "test_rows": len(Xte),
                              "train_start": Xtr.index.min(), "train_end": Xtr.index.max(),
                              "test_start": Xte.index.min(), "test_end": Xte.index.max(),
                              "feature_count": X.shape[1]})
            
            print(f"{target} +{horizon}m: train={len(Xtr):,}, test={len(Xte):,}")
    
    data.to_csv(outdir / "features_15min.csv", index_label="Timestamp")
    
    pd.DataFrame(summaries).to_csv(outdir / "train_test_summary.csv", index=False)
    
    print(f"Saved train/test files in: {outdir.resolve()}")

if __name__ == "__main__": main()


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14272\1251593123.py:30: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  out["DayOfWeek"], out["Day"], out["Month"] = idx.dayofweek, idx.day, idx.month
C:\Users\ASUS\AppData\Local\Temp\ipykernel_14272\1251593123.py:32: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  out["IsWeekend"] = (idx.dayofweek >= 5).astype(int)


Timeline: 2015-05-01 13:30:00 to 2025-12-21 22:30:00
Rows: 373,189; missing intervals: 111,651
Sales Count +15m: train=5,168, test=1,293
Sales Count +30m: train=5,153, test=1,289
Sales Count +60m: train=5,128, test=1,283
Sales Count +120m: train=5,057, test=1,265
Redemption Count +15m: train=5,168, test=1,293
Redemption Count +30m: train=5,153, test=1,289
Redemption Count +60m: train=5,128, test=1,283
Redemption Count +120m: train=5,057, test=1,265
Saved train/test files in: D:\intern\shortFerry_system\dataProcess\forecast_outputs
